# AdS4 Phase-Lock v19: Minimal Separating Probe

**Goal:** extend v18.1 by adding one minimal extra constraint that helps separate solutions which otherwise remain too easy to fit.

This notebook keeps the v18.1 structure:

- shared multi-slice backbone
- EE / WL / GEO / metric outputs
- single-slice baseline
- joint multi-slice training
- held-out interpolation test

and adds one new ingredient:

- **derivative-sensitive constraint**

That gives a simple answer to the next question after v18.1:

> what is the smallest extra structural signal that sharpens identifiability?

## Notebook status

This is a **repo-ready v19 scaffold**.

It uses synthetic placeholder targets so the notebook runs immediately.
When you are ready, replace the target functions with your exact v18.1 / v17 observables.

### v19 change from v18.1
We add derivative matching for:
- EE(x)
- WL(x)
- GEO(x)
- metric(r)

This is the minimal separating probe in this scaffold:
not just values, but local slope structure must align too.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Hyperparameters

In [ ]:
# Control slices
c_values = [0.00, 0.16, 0.30]

# Grids
n_x = 160
n_r = 160
x_min, x_max = -2.0, 2.0
r_min, r_max = 0.2, 3.0

# Training
epochs_joint = 2600
epochs_single = 1800
epochs_heldout = 2200
lr = 1e-3

# Architecture
hidden = 64
depth = 3

# Loss weights
w_ee = 1.0
w_wl = 1.0
w_geo = 1.0
w_metric = 1.0
w_smooth = 1e-4
w_consistency = 0.5

# v19 addition
w_deriv_ee = 0.3
w_deriv_wl = 0.3
w_deriv_geo = 0.3
w_deriv_metric = 0.3

print_every = 250

## Synthetic targets

These are placeholder target families.

The v19 scaffold is designed so that you can later drop in your exact observables from the repo experiments.

In [ ]:
def true_metric(r, c):
    return 1.0 + r**2 - (1.0 + 0.8*c) / (r + 0.25) + 0.06 * c * np.sin(2.5 * r)

def true_ee(x, c):
    return np.log(1.0 + x**2) + 0.10 * c * np.exp(-x**2) + 0.02 * np.cos(2*x)

def true_wl(x, c):
    return np.sqrt(1.0 + x**2) + 0.08 * c / (1.0 + x**2) + 0.015 * np.sin(3*x)

def true_geo(x, c):
    return 1.0 / (1.0 + x**2) + 0.12 * c * x**2 / (1.0 + x**2) + 0.01 * np.cos(4*x)

## Data assembly

In [ ]:
x_grid = np.linspace(x_min, x_max, n_x).astype(np.float32)
r_grid = np.linspace(r_min, r_max, n_r).astype(np.float32)

targets = {}
for c in c_values:
    targets[c] = {
        "x": torch.tensor(x_grid, device=device).unsqueeze(1),
        "r": torch.tensor(r_grid, device=device).unsqueeze(1),
        "ee": torch.tensor(true_ee(x_grid, c), device=device).unsqueeze(1),
        "wl": torch.tensor(true_wl(x_grid, c), device=device).unsqueeze(1),
        "geo": torch.tensor(true_geo(x_grid, c), device=device).unsqueeze(1),
        "metric": torch.tensor(true_metric(r_grid, c), device=device).unsqueeze(1),
    }

print("Slices:", list(targets.keys()))
print("x shape:", targets[c_values[0]]["x"].shape, "| r shape:", targets[c_values[0]]["r"].shape)

## Model

We keep the v18.1 architecture:
- shared observable backbone
- shared metric backbone
- task-specific heads

The v19 change is in the loss, not the architecture.

In [ ]:
class SharedBackbone(nn.Module):
    def __init__(self, in_dim=2, hidden=64, depth=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

class SliceHead(nn.Module):
    def __init__(self, hidden=64, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, h):
        return self.net(h)

class MultiSliceModel(nn.Module):
    def __init__(self, hidden=64, depth=3):
        super().__init__()
        self.backbone_obs = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)
        self.backbone_metric = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)

        self.ee_head = SliceHead(hidden, 1)
        self.wl_head = SliceHead(hidden, 1)
        self.geo_head = SliceHead(hidden, 1)
        self.metric_head = SliceHead(hidden, 1)

    def forward_obs(self, x, c):
        c_tensor = torch.full_like(x, float(c))
        z = torch.cat([x, c_tensor], dim=1)
        h = self.backbone_obs(z)
        return self.ee_head(h), self.wl_head(h), self.geo_head(h)

    def forward_metric(self, r, c):
        c_tensor = torch.full_like(r, float(c))
        z = torch.cat([r, c_tensor], dim=1)
        h = self.backbone_metric(z)
        return self.metric_head(h)

## Loss functions

### v18.1 terms
- value matching
- metric smoothness
- cross-slice latent consistency

### v19 term
- derivative matching

This makes the training sensitive not only to function values, but to local geometric shape.

In [ ]:
mse = nn.MSELoss()

def smoothness_loss(y):
    return ((y[1:] - y[:-1])**2).mean()

def finite_difference(y, x):
    dy = y[1:] - y[:-1]
    dx = x[1:] - x[:-1]
    return dy / (dx + 1e-12)

def derivative_match_loss(pred_y, true_y, x):
    pred_d = finite_difference(pred_y, x)
    true_d = finite_difference(true_y, x)
    return mse(pred_d, true_d)

def backbone_consistency_loss(model, x_probe, c1, c2):
    c1_tensor = torch.full_like(x_probe, float(c1))
    c2_tensor = torch.full_like(x_probe, float(c2))
    z1 = torch.cat([x_probe, c1_tensor], dim=1)
    z2 = torch.cat([x_probe, c2_tensor], dim=1)

    h1 = model.backbone_obs(z1)
    h2 = model.backbone_obs(z2)

    return ((h1 - h2)**2).mean()

def compute_slice_loss_v19(model, batch, c):
    pred_ee, pred_wl, pred_geo = model.forward_obs(batch["x"], c)
    pred_metric = model.forward_metric(batch["r"], c)

    loss_ee = mse(pred_ee, batch["ee"])
    loss_wl = mse(pred_wl, batch["wl"])
    loss_geo = mse(pred_geo, batch["geo"])
    loss_metric = mse(pred_metric, batch["metric"])
    loss_smooth = smoothness_loss(pred_metric)

    loss_deriv_ee = derivative_match_loss(pred_ee, batch["ee"], batch["x"])
    loss_deriv_wl = derivative_match_loss(pred_wl, batch["wl"], batch["x"])
    loss_deriv_geo = derivative_match_loss(pred_geo, batch["geo"], batch["x"])
    loss_deriv_metric = derivative_match_loss(pred_metric, batch["metric"], batch["r"])

    total = (
        w_ee * loss_ee +
        w_wl * loss_wl +
        w_geo * loss_geo +
        w_metric * loss_metric +
        w_smooth * loss_smooth +
        w_deriv_ee * loss_deriv_ee +
        w_deriv_wl * loss_deriv_wl +
        w_deriv_geo * loss_deriv_geo +
        w_deriv_metric * loss_deriv_metric
    )

    parts = {
        "ee": float(loss_ee.item()),
        "wl": float(loss_wl.item()),
        "geo": float(loss_geo.item()),
        "metric": float(loss_metric.item()),
        "smooth": float(loss_smooth.item()),
        "d_ee": float(loss_deriv_ee.item()),
        "d_wl": float(loss_deriv_wl.item()),
        "d_geo": float(loss_deriv_geo.item()),
        "d_metric": float(loss_deriv_metric.item()),
        "total": float(total.item()),
    }
    return total, parts

## Single-slice baseline

In [ ]:
def train_single_slice_v19(c, epochs=epochs_single, lr=lr, hidden=hidden, depth=depth):
    model = MultiSliceModel(hidden=hidden, depth=depth).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    history = {
        "total": [],
        "metric": [],
        "d_metric": [],
    }

    for epoch in range(epochs):
        opt.zero_grad()
        loss, parts = compute_slice_loss_v19(model, targets[c], c)
        loss.backward()
        opt.step()

        history["total"].append(parts["total"])
        history["metric"].append(parts["metric"])
        history["d_metric"].append(parts["d_metric"])

        if epoch % print_every == 0 or epoch == epochs - 1:
            print(f"[single v19 c={c:.2f}] epoch {epoch:4d} | total {parts['total']:.6e} | d_metric {parts['d_metric']:.6e}")

    return model, history

single_models_v19 = {}
single_histories_v19 = {}
for c in c_values:
    print("\nTraining v19 single-slice baseline for c =", c)
    model_c, hist_c = train_single_slice_v19(c)
    single_models_v19[c] = model_c
    single_histories_v19[c] = hist_c

## Joint multi-slice fit

In [ ]:
def train_joint_v19(c_train, epochs=epochs_joint, lr=lr, hidden=hidden, depth=depth, verbose=True):
    model = MultiSliceModel(hidden=hidden, depth=depth).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    history = {
        "total": [],
        "consistency": [],
        "d_metric": {c: [] for c in c_train},
        "slice_total": {c: [] for c in c_train},
    }

    x_probe = torch.linspace(-1.5, 1.5, 100, device=device).unsqueeze(1)

    for epoch in range(epochs):
        opt.zero_grad()

        total_loss = 0.0
        slice_parts = {}

        for c in c_train:
            loss_c, parts = compute_slice_loss_v19(model, targets[c], c)
            total_loss = total_loss + loss_c
            slice_parts[c] = parts

        consistency = 0.0
        for i in range(len(c_train) - 1):
            consistency = consistency + backbone_consistency_loss(model, x_probe, c_train[i], c_train[i+1])

        total_loss = total_loss + w_consistency * consistency
        total_loss.backward()
        opt.step()

        history["total"].append(float(total_loss.item()))
        history["consistency"].append(float(consistency.item()) if torch.is_tensor(consistency) else float(consistency))
        for c in c_train:
            history["slice_total"][c].append(slice_parts[c]["total"])
            history["d_metric"][c].append(slice_parts[c]["d_metric"])

        if verbose and (epoch % print_every == 0 or epoch == epochs - 1):
            print(f"[joint v19] epoch {epoch:4d} | total {history['total'][-1]:.6e} | consistency {history['consistency'][-1]:.6e}")

    return model, history

print("\nTraining joint v19 multi-slice model...")
joint_model_v19, joint_history_v19 = train_joint_v19(c_values)

## Diagnostics helpers

In [ ]:
def metric_mse(model, c):
    with torch.no_grad():
        pred = model.forward_metric(targets[c]["r"], c)
        return float(mse(pred, targets[c]["metric"]).item())

def metric_deriv_mse(model, c):
    with torch.no_grad():
        pred = model.forward_metric(targets[c]["r"], c)
        pred_d = finite_difference(pred, targets[c]["r"])
        true_d = finite_difference(targets[c]["metric"], targets[c]["r"])
        return float(mse(pred_d, true_d).item())

def obs_mse(model, c):
    with torch.no_grad():
        pred_ee, pred_wl, pred_geo = model.forward_obs(targets[c]["x"], c)
        return {
            "ee": float(mse(pred_ee, targets[c]["ee"]).item()),
            "wl": float(mse(pred_wl, targets[c]["wl"]).item()),
            "geo": float(mse(pred_geo, targets[c]["geo"]).item()),
        }

def print_v19_comparison_table():
    print("v19 comparison: single-slice vs joint")
    print("-" * 92)
    print(f"{'c':>6} | {'single metric':>14} | {'joint metric':>14} | {'single d_metric':>16} | {'joint d_metric':>15}")
    print("-" * 92)
    for c in c_values:
        s_metric = metric_mse(single_models_v19[c], c)
        j_metric = metric_mse(joint_model_v19, c)
        s_dmetric = metric_deriv_mse(single_models_v19[c], c)
        j_dmetric = metric_deriv_mse(joint_model_v19, c)
        print(f"{c:6.2f} | {s_metric:14.6e} | {j_metric:14.6e} | {s_dmetric:16.6e} | {j_dmetric:15.6e}")

print_v19_comparison_table()

## Loss curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(joint_history_v19["total"], label="joint total")
plt.plot(joint_history_v19["consistency"], label="consistency")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("v19 joint training losses")
plt.legend()
plt.show()

In [ ]:
for c in c_values:
    plt.figure(figsize=(8, 4))
    plt.plot(single_histories_v19[c]["total"], label=f"single total c={c:.2f}")
    plt.plot(single_histories_v19[c]["d_metric"], label=f"single d_metric c={c:.2f}")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"v19 single-slice losses at c={c:.2f}")
    plt.legend()
    plt.show()

## Held-out interpolation test

Train on outer slices:
- c = 0.00
- c = 0.30

Evaluate on held-out:
- c = 0.16

This is the clearest place to test whether derivative-sensitive structure improves interpolation.

In [ ]:
train_slices = [0.00, 0.30]
heldout_c = 0.16

print(f"Training v19 held-out model on slices {train_slices}, evaluating at held-out c={heldout_c:.2f}")
heldout_model_v19, heldout_history_v19 = train_joint_v19(train_slices, epochs=epochs_heldout, verbose=True)

heldout_metric_err = metric_mse(heldout_model_v19, heldout_c)
heldout_dmetric_err = metric_deriv_mse(heldout_model_v19, heldout_c)
heldout_obs_errs = obs_mse(heldout_model_v19, heldout_c)
heldout_obs_avg = (heldout_obs_errs["ee"] + heldout_obs_errs["wl"] + heldout_obs_errs["geo"]) / 3.0

print("\nHeld-out evaluation (v19)")
print("-" * 60)
print(f"held-out c              : {heldout_c:.2f}")
print(f"metric MSE              : {heldout_metric_err:.6e}")
print(f"metric derivative MSE   : {heldout_dmetric_err:.6e}")
print(f"EE / WL / GEO MSE       : {heldout_obs_errs}")
print(f"observable avg MSE      : {heldout_obs_avg:.6e}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(heldout_history_v19["total"], label="held-out training total")
plt.plot(heldout_history_v19["consistency"], label="held-out consistency")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("v19 held-out interpolation training losses")
plt.legend()
plt.show()

## Held-out observable fits

In [ ]:
x = targets[heldout_c]["x"].detach().cpu().numpy().flatten()

with torch.no_grad():
    pred_ee, pred_wl, pred_geo = heldout_model_v19.forward_obs(targets[heldout_c]["x"], heldout_c)

plt.figure(figsize=(9, 5))
plt.plot(x, targets[heldout_c]["ee"].detach().cpu().numpy(), label="EE true")
plt.plot(x, pred_ee.detach().cpu().numpy(), "--", label="EE pred")
plt.plot(x, targets[heldout_c]["wl"].detach().cpu().numpy(), label="WL true")
plt.plot(x, pred_wl.detach().cpu().numpy(), "--", label="WL pred")
plt.plot(x, targets[heldout_c]["geo"].detach().cpu().numpy(), label="GEO true")
plt.plot(x, pred_geo.detach().cpu().numpy(), "--", label="GEO pred")
plt.xlabel("x")
plt.ylabel("observable")
plt.title("v19 held-out observable fit at c=0.16")
plt.legend()
plt.show()

## Held-out metric fit

In [ ]:
r = targets[heldout_c]["r"].detach().cpu().numpy().flatten()

with torch.no_grad():
    pred_metric = heldout_model_v19.forward_metric(targets[heldout_c]["r"], heldout_c)

plt.figure(figsize=(8, 5))
plt.plot(r, targets[heldout_c]["metric"].detach().cpu().numpy(), label="metric true")
plt.plot(r, pred_metric.detach().cpu().numpy(), "--", label="metric pred")
plt.xlabel("r")
plt.ylabel("metric")
plt.title("v19 held-out metric fit at c=0.16")
plt.legend()
plt.show()

## Held-out derivative fit

This is the main new diagnostic in v19:
not only values, but local slope structure should align.

In [ ]:
with torch.no_grad():
    true_d = finite_difference(targets[heldout_c]["metric"], targets[heldout_c]["r"]).detach().cpu().numpy().flatten()
    pred_d = finite_difference(pred_metric, targets[heldout_c]["r"]).detach().cpu().numpy().flatten()

r_mid = targets[heldout_c]["r"].detach().cpu().numpy().flatten()[1:]

plt.figure(figsize=(8, 5))
plt.plot(r_mid, true_d, label="metric derivative true")
plt.plot(r_mid, pred_d, "--", label="metric derivative pred")
plt.xlabel("r")
plt.ylabel("d(metric)/dr")
plt.title("v19 held-out metric derivative fit at c=0.16")
plt.legend()
plt.show()

## Latent backbone diagnostics

In [ ]:
with torch.no_grad():
    x_probe = torch.linspace(-1.5, 1.5, 120, device=device).unsqueeze(1)
    latents = {}
    for c in c_values:
        c_tensor = torch.full_like(x_probe, float(c))
        z = torch.cat([x_probe, c_tensor], dim=1)
        latents[c] = joint_model_v19.backbone_obs(z).detach().cpu().numpy()

for c in c_values:
    plt.figure(figsize=(9, 4))
    plt.imshow(latents[c].T, aspect="auto", origin="lower")
    plt.xlabel("probe index")
    plt.ylabel("latent channel")
    plt.title(f"v19 joint observable backbone latent map at c={c}")
    plt.colorbar()
    plt.show()

## Interpretation guide

### Success condition
v19 is doing useful work if:
- held-out value fit stays strong
- derivative fit is also strong
- latent maps remain coherent
- metric derivative error is reduced compared with a value-only version

### If derivative matching does not help much
That is still informative:
the ambiguity may require a genuinely new observable,
not just local slope information.

## Suggested next variants

If you want to branch beyond this notebook:

- **v19.1**: derivative-only ablation
- **v19.2**: stronger metric derivative, weaker observable derivatives
- **v19.3**: curvature matching (second derivative)
- **v20**: minimal separating probe sweep

## Final repo note

When moving this into the repo:
1. replace the placeholder target functions  
2. compare directly against v18.1 figures  
3. keep the new derivative-fit plot  
4. summarize whether derivative structure sharpens identifiability enough on its own